In [2]:
import gymnasium as gym 
import random 
import numpy as np 
from collections import deque 
import torch.nn as nn 
import torch 
import torch.nn.functional as F 
import torch.optim as optim 
import matplotlib.pyplot as plt 
import copy


class ReplayBuffer:
    def __init__(self, buffer_size, batch_size):
        self.buffer = deque(maxlen=buffer_size)
        self.batch_size = batch_size

    def add(self, state, action, reward, next_state, done):
        data = (state, action, reward, next_state, done)
        self.buffer.append(data)

    def __len__(self):
        return len(self.buffer)

    def get_batch(self):
        data = random.sample(self.buffer, self.batch_size)

        state = np.stack([x[0] for x in data])
        action = np.array([x[1] for x in data])
        reward = np.array([x[2] for x in data])
        next_state = np.stack([x[3] for x in data])
        done = np.array([x[4] for x in data], dtype=np.int32)

        return state, action, reward, next_state, done


env = gym.make("CartPole-v1", render_mode="human")
replay_buffer = ReplayBuffer(buffer_size=10000, batch_size=32)

try:
    episode = 0

    while episode < 10 or len(replay_buffer) < replay_buffer.batch_size:
        state, info = env.reset()
        done = False

        while not done:
            action = env.action_space.sample()

            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

            replay_buffer.add(
                state,
                action,
                reward,
                next_state,
                done
            )

            state = next_state

        episode += 1

finally:
    env.close()


state, action, reward, next_state, done = replay_buffer.get_batch()

print(state.shape)       # (32, 4)
print(action.shape)      # (32,)
print(reward.shape)      # (32,)
print(next_state.shape)  # (32, 4)
print(done.shape)        # (32,)

(32, 4)
(32,)
(32,)
(32, 4)
(32,)


In [3]:
class QNet(nn.Module):
    def __init__(self, action_size):
        super().__init__()

        self.l1 = nn.Linear(4, 128)
        self.l2 = nn.Linear(128, 128)
        self.l3 = nn.Linear(128, action_size)

    def forward(self, x):
        x = F.relu(self.l1(x))
        x = F.relu(self.l2(x))
        x = self.l3(x)

        return x
    
class DQNAgent:
    def __init__(self):
        self.gamma = 0.98
        self.lr = 0.0005
        self.epsilon = 0.1
        self.buffer_size = 10000
        self.batch_size = 32
        self.action_size = 2

        self.replay_buffer = ReplayBuffer(
            self.buffer_size,
            self.batch_size
        )

        self.qnet = QNet(self.action_size)
        self.qnet_target = QNet(self.action_size)

        self.optimizer = optim.Adam(
            self.qnet.parameters(),
            lr=self.lr
        )

    def get_action(self, state):
        if np.random.rand() < self.epsilon:
            return np.random.choice(self.action_size)

        else:
            state = state[np.newaxis, :]
            state = torch.tensor(state, dtype=torch.float32)

            with torch.no_grad():
                qs = self.qnet(state)

            return qs.argmax().item()

    def update(self, state, action, reward, next_state, done):
        self.replay_buffer.add(
            state,
            action,
            reward,
            next_state,
            done
        )

        if len(self.replay_buffer) < self.batch_size:
            return

        state, action, reward, next_state, done = self.replay_buffer.get_batch()

        state = torch.tensor(state, dtype=torch.float32)
        action = torch.tensor(action, dtype=torch.long)
        reward = torch.tensor(reward, dtype=torch.float32)
        next_state = torch.tensor(next_state, dtype=torch.float32)
        done = torch.tensor(done, dtype=torch.float32)

        qs = self.qnet(state)

        q = qs[torch.arange(self.batch_size), action]

        with torch.no_grad():
            next_qs = self.qnet_target(next_state)
            next_q = next_qs.max(dim=1).values

            target = reward + (1 - done) * self.gamma * next_q

        loss = F.mse_loss(q, target)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

    def sync_qnet(self):
        self.qnet_target.load_state_dict(self.qnet.state_dict())

In [3]:
episodes = 300
sync_interval = 20  # 신경망 동기화 주기(20번째 에피소드마다 동기화)
env = gym.make("CartPole-v1", render_mode="rgb_array")
agent = DQNAgent()
reward_history = []  # 에피소드별 보상 기록

for episode in range(episodes):
    state = env.reset()[0]
    done = False

    total_reward = 0

    while not done:
        action = agent.get_action(state)
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated | truncated

        agent.update(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward

    if episode % sync_interval == 0:
        agent.sync_qnet()

    reward_history.append(total_reward)

KeyboardInterrupt: 